# Lesson 21 · 🎨 Matrices as filters — your headphones do matrix multiplication every second: the DFT matrix and the mel matrix, audio as linear algebra

Aim the view of *The Art of Linear Algebra* at the Audio Core:
**the DFT is one matrix × vector; mel features are one matrix × vector.** Once you see this, the FFT and mel filtering stop being "black-box formulas" and become linear algebra you already know.

**Goal**: write the DFT and the mel transform each in matrix form, and check that `W @ x` agrees numerically with `fft`.

> **Success criterion for this lesson**: recognize that "what happens in your headphones is a matvec" — you are **not** required to derive the full DFT proof by hand (that's L37's job).

**Why it matters for Aurora**: from the matrix point of view, the FFT is a fast algorithm for the DFT matrix, and mel filtering is another matrix multiplying the spectrum from the left.

**Linear-algebra module recap (one line)**: vectors → dot products/norms → matrix multiplication → special matrices → eigen/SVD → linear systems and invertibility → **the DFT/mel matrices in audio (this lesson)**.

← **Previous lesson** · [L20 · A map of decompositions](L20_visual_factorizations.ipynb)

> Last lesson covered **a map of decompositions**: the geometric meaning of the LU / QR / SVD decompositions, compared.
> This lesson explores **matrices as filters**.

## This lesson's story: matrices are the sound you hear every day

Every time you play a song, your phone does this:
1. Cut the audio into frames → each frame is a vector `x` of length N
2. Do an FFT: `X = W · x`, where W is an N×N complex matrix
3. Do mel filtering: `m = M · X_power`, where M is a real (n_mels × N//2+1) matrix

Both steps are **matrix × vector**. No "black box" anywhere — just multiplying by coefficients and adding.

`W[k,n] = e^{-2πi·kn/N}` is the (k, n) entry of the DFT matrix.
You can skip `np.fft.fft` and build one row of the matrix yourself with `np.exp` to do the same job.

Each row of the mel matrix `M` is a triangular filter, and `M @ p` is the matrix returned by `mel_filterbank()` taking one weighted sum of `p`.
In Aurora the two are wrapped as `fft()` and `mel_filterbank()`, and underneath both are weighted dot products.

This lesson's tasks:
- Build the 8×8 DFT matrix W explicitly and check `W @ x == np.fft.fft(x)`
- Understand the shape of the mel matrix: each row of `M` holds one band's weighting coefficients
- Implement `build_dft_matrix(N)` and pass the check

## A 2-minute review before we start: W @ x really is the DFT

- `x` is a signal vector of length N
- `W` is the N×N DFT matrix
- each row of `W @ x` is one weighted sum, corresponding exactly to one frequency bin

L37 will swap this for a hand-written loop; for now, lock in the matrix view.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
from laviz import style, heatmap, mat_times_vec
from aurora.audio import fft, mel_filterbank, sine
style(); print('Ready')

## Reading the notation: shape first, then the operation

The signal vector `x` has shape `(N,)`, the DFT matrix `W` is `(N, N)`, and the mel matrix `M` is `(n_mels, N//2+1)`. Matrix multiplication recombines the components of the input linearly — each output value is one dot product.

## Sharpening the axe: where does the DFT formula `exp(-2πi·kn/N)` come from?

Before building the matrix for real, spend a few minutes taking this formula apart — it isn't a random pile of symbols; every part has a concrete meaning.

**A story first**: picture a circular running track with N people standing evenly around it, numbered n = 0, 1, …, N−1. "Frequency k" is like a runner who goes k times faster than everyone else — by the time the frequency-1 runner has done one full lap (2π radians), the frequency-k runner has done k laps.

- `n/N` is "what fraction of a full lap the frequency-1 runner has covered on reaching position n" (a "number of laps").
- Multiply by the k-times speed, and "how many laps the frequency-k runner has covered at the same position" is `k·n/N`.
- Multiply by 2π again to convert "laps" into "angle in radians" — just as converting "hours" to "minutes" means multiplying by 60, converting "laps" to "radians" means multiplying by 2π (one full lap = 2π radians; that's how radians are defined).

So `2π·kn/N` is the rotation angle for "frequency k, position n" — this is the `twiddle(k,n,N) = e^{-2πikn/N}` (twiddle factor) from L06 making another appearance.

**And the minus sign in front?** What the DFT does is "read off how much of a given frequency is in the signal", and mathematically that step takes the **inner product** of the signal vector `x` with the "reference waveform" `e^{+i2πkn/N}` — and the definition of the complex inner product takes the **conjugate** of the second vector (`⟨a,b⟩ = Σ a[n]·conj(b[n])`). The conjugate of `e^{+iθ}` is exactly `e^{-iθ}` (flip the sign of the imaginary part — the "conjugate = reflect across the horizontal axis" from L05). So the conjugation in the "read off a frequency component" step automatically turns the exponent negative — not an arbitrary convention, but something the act of "taking an inner product" demands. Going the other way, "synthesizing the original signal from its frequency components" (the IDFT) uses the plus sign, and the inverse of W is the conjugate transpose `Wᴴ/N` (we'll check this shortly).

Below we work through the row N=4, k=1 by hand, turning the story above into numbers you can see.

In [ ]:
import numpy as np

# Check by hand: the row N=4, k=1 — 4 people split one full lap, the frequency-1 runner goes once around
N4 = 4
k1 = 1
n4 = np.arange(N4)
angles = 2 * np.pi * k1 * n4 / N4        # the rotation angle at each position (radians)
print('angles (radians) =', np.round(angles, 4), ' i.e.', np.round(angles / np.pi, 2), '× π')
row1 = np.exp(-1j * angles)
print('row 1 of W (k=1) =', np.round(row1, 4))
print('Should be [1, -j, -1, j] by hand?', np.allclose(row1, [1, -1j, -1, 1j]))

### Extra: why does `np.outer(n, n)` produce every `k·n` in one go?

`np.outer(a, b)` is called the **outer product**: given two vectors a, b of length N, it returns an N×N matrix whose (i, j) entry is `a[i] * b[j]` — like the times table from school: a down the side, b across the top, and each box filled with the product of the two.

```python
np.outer([1,2,3], [1,2,3])
# array([[1, 2, 3],
#        [2, 4, 6],
#        [3, 6, 9]])
```
The value in row 2, column 3 (counting from 0, i.e. i=1, j=2) is `2×3=6`, exactly "the i-th number × the j-th number".

Back to the DFT: entry (k,n) of the matrix should hold `exp(-2πi·k·n/N)`, and the `k·n` part is just the times table of "frequency index k" and "position index n" — compute every k·n at once with `np.outer(k_idx, n_idx)`, multiply the whole thing by `-2πi/N`, and finally take `np.exp` of each box (NumPy works elementwise automatically), and you've assembled the full W matrix without writing a double for loop.

In [ ]:
demo_outer = np.outer([1, 2, 3], [1, 2, 3])
print(demo_outer)
print('entry (1,2) =', demo_outer[1, 2], ' should equal 2*3 =', 2 * 3)

## Building the DFT matrix

Run the cell below and watch the DFT matrix `W` being built: `W[k,n] = exp(-2πi·kn/N)`.
Once it's built, check its **orthogonality**: `W @ W^H / N = I`, with a maximum error far below machine precision.
What this step means: the FFT is, at heart, a matrix multiplication on the signal vector; the butterfly recursion just folds O(N²) down to O(N log N).

In [ ]:
import numpy as np

# DFT matrix W_N: W[k,n] = exp(-2πi k n / N)
N = 8
n = np.arange(N)
W = np.exp(-2j * np.pi * np.outer(n, n) / N)  # shape (N,N)
print(f'DFT matrix W_{N} shape = {W.shape}')
print(f'Rows of W: orthogonality check  W @ W^H / N = I')
ortho_err = np.max(np.abs(W @ W.conj().T / N - np.eye(N)))
print(f'max error = {ortho_err:.2e}  (should be close to 0)')
# compare with numpy.fft.fft
x = np.array([1.,2.,3.,4.,5.,6.,7.,8.])
print(f'W@x ≈ np.fft.fft(x) ? {np.allclose(W @ x, np.fft.fft(x))}')


## Looking ahead · L37 will write the same transform as a loop (2 minutes)

This lesson uses the matrix form: `X = W @ x`, numerically equal to `np.fft.fft(x)` (`atol=1e-10`).

**L37** will write it another way: no whole matrix, just a double loop that writes out
`X[k] = Σₙ x[n]·e^{-2πikn/N}` directly — the two are equivalent;
the matrix version helps you remember "FFT = linear transformation", the loop version helps you remember each exponential term.


### Extra: how exactly does the FFT turn O(N²) into O(N log N)?

The matrix product `W @ x` has to produce N outputs, each a dot product of length N (N multiplications), for a total of `N × N = N²` multiplications — that's why the matrix version is slow.

The FFT's core trick is called **divide and conquer**:

1. Split the length-N signal in half by even and odd index — the even indices `x[0], x[2], x[4], …` and the odd indices `x[1], x[3], x[5], …`, each half of length N/2.
2. The algebra of the DFT formula itself guarantees: **the result of a length-N DFT can be assembled from just two length-N/2 DFT results** (with some "twiddle factors" for fine adjustment), without doing another N/2×N/2 matrix multiplication.
3. So "compute 1 DFT of length N" becomes "compute 2 DFTs of length N/2, then spend O(N) time stitching them together".
4. This split can recurse all the way down: N/2 splits into 2 of N/4, N/4 into 2 of N/8… until the length is 1 (nothing to compute; just return itself).

The number of levels of splitting is `log₂N` (each level halves the length; however many halvings it takes to reach 1 is the number of levels); each level stitches all the sub-results together for a total of only O(N) operations. So total time = O(N) per level × log₂N levels = **O(N log N)**. That's where the name "butterfly" comes from — the step where two sub-results are paired up and merged via a "twiddle factor" looks like a pair of butterfly wings when drawn. L37 will write this whole process out with loops.

In [ ]:
for Ndemo in [8, 64, 512, 4096]:
    naive_ops = Ndemo ** 2
    fft_ops = int(Ndemo * np.log2(Ndemo))
    print(f'N={Ndemo:5d}  matrix version≈{naive_ops:>9d} multiplications   FFT version≈{fft_ops:>7d}   speed-up≈{naive_ops/fft_ops:.1f}x')

## A simplified mel filter: building the triangle matrix by hand

Run the cell below to see a hand-built 3-filter mel matrix (over 8 FFT bins):
each row is a triangular filter, peaking at its center frequency and falling linearly to 0 on both sides.
`M @ mag` compresses the power spectrum vector into a 3-dimensional vector of mel energies — this is the core operation of mel feature extraction.

### Extra: why use `N_mels+2` endpoints rather than `N_mels`?

Each triangular filter needs three "anchor points" to fix its shape: **left foot (value 0) → peak (value 1) → right foot (value 0)** — a single filter on its own needs 3 points.

But neighboring filters stand "shoulder to shoulder" and share endpoints: the "right foot" of filter 1 is the "left foot" of filter 2… The construction here also requires each filter to fix both its "rising slope" and its "falling slope", using a window of three neighboring endpoints: filter `m` uses `centers[m]` (left foot), `centers[m+1]` (peak) and `centers[m+2]` (right foot).

For the last filter (filter `N_mels-1`) to reach `centers[N_mels+1]` as well, the endpoint array must have length `N_mels + 2` — not an arbitrary number, but a direct consequence of the geometry "each filter takes 3 consecutive endpoints, and neighboring filters share endpoints". (You could pad the edges another way, such as adding a virtual mirror point at each end, with equivalent results; this is just the most direct choice, with no extra assumptions.)

Below we count it out with `N_mels=3`: there should be 5 endpoints.

In [ ]:
N_mels_demo = 3
centers_demo = np.linspace(0, 4, N_mels_demo + 2)
print('centers =', centers_demo, ' length =', len(centers_demo), '(= N_mels+2 =', N_mels_demo + 2, ')')
for m in range(N_mels_demo):
    lo, cen, hi = centers_demo[m], centers_demo[m + 1], centers_demo[m + 2]
    print(f'filter {m}: left foot={lo}, peak={cen}, right foot={hi}')

In [ ]:
import numpy as np

# Mel filter bank = matrix multiplication M @ |X|²
# Simplified demo: build 3 triangular filters by hand (over 8 FFT bins)
N_fft, N_mels = 8, 3
bins = np.arange(N_fft // 2 + 1, dtype=float)  # [0, 1, 2, 3, 4]
# Fix: use N_mels+2 endpoints (including the boundaries 0 and N_fft//2)
# Each filter m uses the neighboring triple (centers[m], centers[m+1], centers[m+2]) as lo/cen/hi
centers = np.linspace(0, N_fft//2, N_mels+2)  # [0.0, 1.0, 2.0, 3.0, 4.0]
M = np.zeros((N_mels, len(bins)))
for m in range(N_mels):
    lo, cen, hi = centers[m], centers[m+1], centers[m+2]
    M[m] = np.clip(np.where(bins<=cen, (bins-lo)/(cen-lo+1e-9),
                             (hi-bins)/(hi-cen+1e-9)), 0, 1)
print(f'Mel matrix M shape = {M.shape}  (N_mels × N_fft//2+1)')
print(np.round(M, 2))
mag = np.array([0.1, 0.5, 1.0, 0.8, 0.3])  # simulated power spectrum |X|²
mel_energy = M @ mag
print(f'Mel energies = {np.round(mel_energy, 4)}')
assert M[0].sum() > 0, 'boundary fix failed: filter 0 should not be all zeros'
print('✅ mel matrix boundary check passed (filter 0 is not all zeros)')


### Extra: what is the `+1e-9` for? Why do too many filters produce all-zero rows?

In the code above, the `1e-9` in `(cen-lo+1e-9)` and `(hi-cen+1e-9)` is a **tiny-value guard** (often called epsilon): it exists purely to stop the program from blowing up when a denominator happens to be exactly 0 (dividing by 0 in floating point gives `inf` or `nan`). Normally `cen` and `lo` are two different points and the denominator is never really 0, and then adding `1e-9` affects the result only at the 1e-9 level — small enough to ignore completely; it **does not "contaminate" normal results**, it only prevents errors in extreme cases.

So how do "lots of all-zero filters" happen? — when **the number of filters (n_mels) far exceeds the number of available frequency bins**. For example, `n_fft=64` gives only 33 frequency bins (`n_fft//2+1`); cram 128 filters in and, for many of them, the spacing between "left foot – peak – right foot" is smaller than one bin's width — so no integer bin can fall inside the overlap of "rising edge ≥ 0" and "falling edge ≥ 0", `min(rising, falling)` comes out negative at every bin, `clip(...,0,None)` cuts it straight to 0, and the whole row collapses to zeros.

This isn't a bug; it's the inevitable result of "not enough resolution": **the number of frequency bins sets how many non-overlapping filters you can meaningfully cut**. The rule of thumb is to keep `n_mels` well below `n_fft//2+1` (a factor of 5–10 or more is safe). Below, Aurora's real `mel_filterbank()` reproduces this degenerate case:

In [ ]:
M_degenerate = mel_filterbank(n_mels=128, n_fft=64, sample_rate=16000)
zero_rows = np.sum(np.all(M_degenerate == 0, axis=1))
print(f'n_mels=128, n_fft=64 → number of frequency bins = {M_degenerate.shape[1]}')
print(f'all-zero filter rows = {zero_rows} / {M_degenerate.shape[0]}')
print('Reason: the number of bins (33) is far smaller than the number of filters (128); many filters crowd onto the same bin and their weights collapse to all zeros.')

## 1. The DFT is a matrix W

Why is the FFT a matrix multiplication?

The ordinary definition of the Fourier transform: `X[k] = Σₙ x[n] · e^{-2πi·kn/N}`
Each output `X[k]` is a **weighted sum** of the inputs `x[n]` — that's one dot product.

Stack N dot products into a matrix and you get the **DFT matrix** W:

$$W[k,n] = e^{-2\pi i \cdot kn/N}, \quad k,n = 0,1,\ldots,N-1$$

Worked example (N=4, k=1, n=2): `W[1,2] = exp(-2πi·2/4) = exp(-πi) = -1`

Properties:
- **Unitary** (up to scale): `W·Wᴴ = N·I`, i.e. `(1/√N)·W` is a unitary matrix
- **Symmetric**: `W[k,n] = W[n,k]` (the DFT matrix equals its own transpose)
- **Inverse transform**: the IDFT matrix = `Wᴴ/N` (conjugate transpose divided by N)

So `W @ x` is exactly equivalent to `np.fft.fft(x)` — we've just written the hidden matrix out explicitly.

### Sharpening the axe: remember Euler's formula from L06? — a unit-circle cheat sheet

From L06: `e^{iθ} = cosθ + i·sinθ`, i.e. "angle θ corresponds to the point (cosθ, sinθ) on the unit circle". Here's a cheat sheet of common angles; the whiteboard challenge uses it directly:

| θ (radians) | θ (degrees) | cos θ | sin θ | e^{iθ} |
|---|---|---|---|---|
| 0 | 0° | 1 | 0 | 1 |
| π/6 | 30° | √3/2 | 1/2 | √3/2 + i/2 |
| π/4 | 45° | √2/2 | √2/2 | √2/2 + i√2/2 |
| π/3 | 60° | 1/2 | √3/2 | 1/2 + i√3/2 |
| π/2 | 90° | 0 | 1 | i |
| π | 180° | −1 | 0 | −1 |
| 3π/2 | 270° | 0 | −1 | −i |
| 2π | 360° | 1 | 0 | 1 |

**Working the whiteboard problem step by step**: `W[2,3] = exp(-2πi·2·3/8) = exp(-2πi·6/8) = exp(-3πi/2)`

- Step 1: simplify the angle. `-2π·6/8 = -12π/8 = -3π/2` (radians).
- Step 2: convert `-3π/2` to degrees: `-3π/2 = -270°`, which is equivalent to `+90°` (a full 360° turn leaves the position unchanged: `-270°+360°=90°`).
- Step 3: look it up in the table: `90°` gives `cos=0, sin=1`.
- Step 4: `e^{-i3π/2} = e^{i(-3π/2 + 2π)} = e^{iπ/2} = cos(π/2)+i·sin(π/2) = 0 + i·1 = i` (that is, `j`).

The code below checks all four steps.

In [ ]:
# Step-by-step check that exp(-3πi/2) = j
theta = -3 * np.pi / 2
theta_equiv = theta + 2 * np.pi            # one full turn, an equivalent angle
print(f'original angle = {theta:.4f} rad = {np.degrees(theta):.1f}°')
print(f'equivalent angle after adding 2π = {theta_equiv:.4f} rad = {np.degrees(theta_equiv):.1f}°')
print(f'cos = {np.cos(theta):.4f}, sin = {np.sin(theta):.4f}')
result = np.cos(theta) + 1j * np.sin(theta)
print(f'e^(-i3π/2) = {result:.4f}  (should be 0+1j, i.e. j)')
assert np.isclose(result.real, 0) and np.isclose(result.imag, 1)
print('✅ matches the answer to whiteboard Q1')

### Extra: what exactly is Wᴴ (the conjugate transpose)? Why does the "orthogonality check" use it, and divide by N too?

**Start with a single complex number**: the squared "length" (modulus) of a complex number `z = a+bi` is `|z|² = a²+b²`. How do you get that algebraically? — multiply z by its **conjugate** `z̄ = a-bi`: `z·z̄ = (a+bi)(a-bi) = a²+b² = |z|²`. Plain `z·z` (no conjugate) gives `a²-b²+2abi`, which is generally a complex number, not a "length". The **conjugate** (flipping the sign of the imaginary part) is the key move that makes complex multiplication produce a "real length".

The same trick carries straight over to vectors: for a vector `v` with complex entries, its "squared length" is `v^H v` (conjugate each component, transpose into a row vector, then dot with itself), not the ordinary `v^T v`. Here `v^H` (read "the conjugate transpose of v", Hermitian transpose) = conjugate first, then transpose; in code it's `v.conj().T`.

**Back to the DFT matrix**: the step `W @ W.conj().T` is computing "the complex inner product of each row of W with itself (and with the other rows)". If all the rows are pairwise orthogonal and every row has "length" `√N` (each row holds N twiddle factors of modulus 1, so its squared length is N), then `W @ Wᴴ` should come out as `N·I` — the diagonal is each row's own squared length N, and the off-diagonal entries (inner products of different rows) should be 0. **Dividing by N** just cancels out the "length is √N" part, so the result becomes the standard identity matrix `I`, lining up neatly with the concept of "orthogonal".

**"Unitary" and "orthogonal" are two names for the same concept in the real and complex worlds**:
- A real matrix Q with `Qᵀ Q = I` is called **orthogonal** (the transpose is enough, because a real number's conjugate is itself).
- A complex matrix U with `Uᴴ U = I` is called **unitary** (you must use the conjugate transpose, for the reason in the "complex length" paragraph above).

`(1/√N)·W` is a unitary matrix — not a happy accident, but a property that settles two very practical things:
1. **Inverting costs almost nothing**: inverting a general matrix is expensive (you have to solve a linear system), but the inverse of a unitary matrix is just its conjugate transpose, here `Wᴴ/N` — which is why the IDFT (inverse transform) can use `Wᴴ/N` directly, without computing an inverse separately.
2. **Energy is neither lost nor gained (Parseval's theorem)**: a signal's "energy" in the time domain (`Σ|x[n]|²`) equals its "energy" in the frequency domain (`Σ|X[k]|²/N`) — guaranteeing that an FFT/IFFT never loses or creates energy out of thin air, which is the fundamental reason we can later transform back and forth with confidence when analyzing spectra and designing filters.

While we're at it, let's check below: with only the **ordinary transpose** (no conjugate), `W @ W.T / N` does not equal the identity matrix — the conjugation step is essential and can't be skipped.

In [ ]:
N8 = 8
k_idx = np.arange(N8)[:, None]
n_idx = np.arange(N8)[None, :]
W8 = np.exp(-2j * np.pi * k_idx * n_idx / N8)

conj_result = W8 @ W8.conj().T / N8
plain_result = W8 @ W8.T / N8

print('conjugate-transpose version W@Wᴴ/N ≈ I ?', np.allclose(conj_result, np.eye(N8)))
print('ordinary-transpose version  W@Wᵀ/N ≈ I ?', np.allclose(plain_result, np.eye(N8)))
print('\nDiagonal of the ordinary-transpose version (mostly not 1, clearly not the identity — proof that the conjugate is required):')
print(np.round(np.diag(plain_result).real, 3))

In [ ]:
N = 8
k = np.arange(N)[:, None]; n = np.arange(N)[None, :]
W = np.exp(-2j*np.pi*k*n/N)        # DFT matrix
heatmap(W.real, title='Real part of the DFT matrix W (N=8)', cmap='coolwarm');
heatmap(W.imag, title='Imaginary part of the DFT matrix W (N=8)', cmap='coolwarm');

## 2. Check: `W·x` matches our hand-written FFT exactly

`W @ x` and the `fft()` in `src/aurora/audio/transforms.py` compute the same transform: the first is an explicit O(N²) matrix multiplication, the second an O(N log N) butterfly recursion, and their numerical results differ by no more than floating-point error (`atol=1e-10`). `mel_filterbank()` in `src/aurora/audio/mel.py` is the same story — it returns the filter matrix `M` directly, and so-called "mel feature extraction" is a single `M @ p` (`p` is the power spectrum), with no other hidden steps.

In [ ]:
x = sine(1.0, duration=N/8, sample_rate=8)[:N]   # any signal of length N
X_matrix = W @ x          # matrix × vector
X_fft = fft(x)            # our hand-written FFT
print('W·x matches fft(x):', np.allclose(X_matrix, X_fft))
assert np.allclose(X_matrix, X_fft, atol=1e-10)
print('✅ What an FFT really is = one matrix × vector (just computed faster).')

## 3. The mel filterbank is a matrix too

mel features = `M · p`, where `M` is the (n_mels × n_freq) filter matrix and `p` is the power spectrum vector. Plot M — a neat row of triangular filters.

### Extra: why are the filters "triangles" rather than rectangles or Gaussians?

First, a fact about the human ear: our perception of frequency isn't "linear" but closer to "logarithmic" — you can easily tell 100 Hz from 200 Hz (only 100 Hz apart), yet struggle to tell 5000 Hz from 5100 Hz (also 100 Hz apart). Psychoacoustic measurements come out roughly as "fine resolution at low frequencies, coarse at high frequencies", and the mel scale is a new scale made by stretching/compressing the frequency axis to follow this pattern (spreading out the low end, compressing the high end).

So why pick a triangle for the filter shape, rather than a rectangle (box) or a Gaussian (bell curve)?
- **Rectangle**: a hard cut — a bin either counts 100% toward the band or 0%, so a small frequency wobble at the edge makes the energy "jump" into the neighboring band; not smooth enough.
- **Gaussian**: smooth in shape, but it needs an exponential function, theoretically extends forever (no clean edge), costs more to compute, and leaves you to decide "how wide is wide enough".
- **Triangle**: the compromise — two straight lines (linear interpolation) are enough to give the center frequency the largest weight and fall smoothly to 0 on both sides, using only add/subtract/multiply/divide, with naturally clean edges (strictly 0 outside the range). That's why almost every speech recognition system uses a triangular mel filter bank: **cheap, good enough, and barely different from a Gaussian in practice**.

In one line: the triangle wasn't chosen because it's "the closest" to the ear's response curve, but as an engineering compromise — "the simplest shape, in exchange for a good-enough approximation of hearing". Below, a direct comparison of the weights of a rectangular filter and a triangular filter over the same set of frequency bins:

In [ ]:
bins_demo = np.arange(20, dtype=float)
rect = np.where((bins_demo >= 5) & (bins_demo < 10), 1.0, 0.0)             # rectangle: hard cut
tri = np.clip(np.where(bins_demo <= 7.5, (bins_demo - 5) / 2.5, (10 - bins_demo) / 2.5), 0, 1)  # triangle: linear transition
print('rectangular filter weights:', np.round(rect, 2))
print('triangular filter weights :', np.round(tri, 2))
print('The rectangle jumps straight from 0 to 1 at bin=4→5; the triangle climbs smoothly 0→0.2→0.4….')

In [ ]:
M = mel_filterbank(n_mels=10, n_fft=64, sample_rate=16000)
print('mel filter matrix shape:', M.shape, '= (n_mels, n_freq_bins)')
heatmap(M, title='mel filter bank = one matrix (one triangular filter per row)', cmap='magma');

In [ ]:
# Check: what mel_filterbank() returns is the matrix M — mel features = M @ p, nothing more
p_full = np.abs(fft(sine(440., duration=64/16000, sample_rate=16000))[:M.shape[1]])**2
M_ref = mel_filterbank(n_mels=10, n_fft=64, sample_rate=16000)
assert p_full.shape == (M_ref.shape[1],), f'shape mismatch: {p_full.shape}'
max_err = np.max(np.abs(M @ p_full - M_ref @ p_full))  # the M from the previous cell and the freshly built M_ref should agree entry by entry
assert np.allclose(M, M_ref, atol=1e-12), 'two constructions of mel_filterbank disagree'
print('mel filter matrix output:', np.round(M_ref @ p_full, 6))
print('✅ mel_filterbank() output is deterministic: both constructions give the same M, mel features = M @ p, max error:', max_err)


## 4. "Each mel value = one weighted dot product over the frequency bins" (the Mv1 view)

Take a small example: multiply the power spectrum `p` by the first few rows of the filter matrix, and use `mat_times_vec` to draw it as "matrix × vector". The picture shows both readings at once: **the top row (Mv1)** is each row · `p` as a dot product — the reading this section is about, where one row of filter weights dotted with the power spectrum gives one mel value; **the bottom row (Mv2)** is a linear combination of the columns, another way of seeing the same result (the two views from L19).

In [ ]:
rng = np.random.default_rng(0)
Msmall = np.round(M[:3, :5], 2)         # take 3 filters × 5 frequency bins
p = np.round(rng.random(5), 2)          # a pretend power spectrum
mat_times_vec(Msmall, p);               # each mel = one row · p = a dot product

**🎉 Wrapping it together**:
- **FFT** = matrix `W` × signal (the `v1` Mv picture)
- **mel features** = matrix `M` × power spectrum
- **neural-network layer** = matrix `W` × input

The same "matrix × vector" picture runs from the Audio Core all the way to deep learning. This is how **visual thinking** welds linear algebra and Aurora together.

In [ ]:
import numpy as np

# Simplified DCT demo (not a full MFCC): showing the matrix-chain view
# Full MFCC = signal → W(DFT) → spectrum → M(mel) → log → D(DCT) → MFCC
# Note: this cell skips the mel and log steps, only demonstrates the matrix-chain structure, and takes only the first N//2+1 independent bins
N = 16  # frame length
n = np.arange(N)
W = np.exp(-2j * np.pi * np.outer(n, n) / N)  # DFT
n_half = np.arange(N // 2 + 1)                # the first N//2+1 = 9 independent frequency bins
N_half = len(n_half)
DCT_row = lambda ki: np.cos(np.pi * (2*n_half + 1) * ki / (2 * N_half))
D = np.array([DCT_row(ki) for ki in range(4)])  # the first 4 DCT coefficients, acting on 9 bins
print(f'W shape = {W.shape}')
print(f'D(4 MFCC) shape = {D.shape}')
x = np.random.default_rng(0).standard_normal(N)
spectrum = np.abs(W @ x)**2  # power spectrum
mfcc4 = D @ spectrum[:N//2+1]  # first 4 DCT orders (simplified: skips mel; independent bins only)
print(f'mfcc (first 4 orders) = {np.round(mfcc4, 4)}')


### Extra: what exactly does this "simplified DCT demo" simplify? Can it be compared with a real MFCC?

The full MFCC extraction pipeline has five steps:
`signal x → DFT(W) → power spectrum |X|² → mel filtering(M) → log → DCT(D) → MFCC`

The code cell above deliberately skips the "mel filtering" and "log" steps and applies the DCT straight to the power spectrum. That's so you can first see one thing clearly — **an MFCC is, at heart, several matrices multiplied in sequence** (W then M then D, a "matrix chain"); what each step does in detail, and why the log is needed (it models the ear's logarithmic perception of loudness), isn't the focus of this lesson — the later audio lessons cover it properly.

So: **the `mfcc4` computed in this cell cannot be compared numerically with a real `mel_spectrogram`/MFCC result** — they differ, because the real pipeline has two more nonlinear steps, mel compression and log compression. This cell's value is only in its "structure": it shows you the skeleton of "signal in, matrix chain, coefficients out". Below is an extra comparison using Aurora's real mel filters + log, to see how far apart the "skip-step version" and a "more complete version" are numerically:

In [ ]:
from aurora.audio import mel_filterbank as _mel_fb

M_full = _mel_fb(n_mels=8, n_fft=N, sample_rate=16000)      # N=16, from the previous cell
mel_energy_full = M_full @ spectrum[:N // 2 + 1]
log_mel_full = np.log(mel_energy_full + 1e-9)
print('skip-step mfcc4 (no mel, no log)              :', np.round(mfcc4, 4))
print('more complete log-mel energies (8 mel bands)  :', np.round(log_mel_full, 4))
print('The two differ in numerical range and in physical meaning — the skip-step version only shows the "skeleton" of the matrix chain; it is not a real MFCC.')

## Parameter experiment: turn one knob at a time

Change `mel_filterbank`'s `n_mels` from `80` to `40`: the shape of the mel matrix `M` goes from `(80, F)` to `(40, F)`, and the final feature vector's dimension halves. Fewer filters means neighboring frequency bins get merged into the same triangular window, lowering high-frequency detail resolution. Going the other way, raise `n_mels` to `128` and watch how the filter overlap changes as the matrix gains rows, and whether the length of the output vector `M @ p` grows in step.

In [ ]:
# Parameter experiment: change n_mels and watch the shape of the mel matrix change
for nm in [40, 80, 128]:
    M_exp = mel_filterbank(nm, 64, 16000)
    print(f'n_mels={nm:3d}  →  mel matrix shape = {M_exp.shape}  ')
# larger n_mels → more triangular windows → higher output feature dimension (finer high-frequency resolution, more matrix rows)


### Extra: how can you "see" how much the filters overlap?

"Overlap" means: do two neighboring triangular filters both have non-zero values on the same frequency bin? Quantifying it is straightforward — for each pair of neighboring filters (row m and row m+1), count how many bins satisfy "both rows > 0", and divide by the total number of bins one of the filters covers, to get an "overlap ratio". The more filters (the larger n_mels), the narrower each triangle, but because the total frequency range stays the same, the overlap ratio between neighbors usually stays at a similar level — that's a design feature of the mel filter bank: each filter overlaps its left and right neighbors by about half, guaranteeing that no point of the power spectrum has its energy "left out".

In [ ]:
def overlap_ratio(M_in):
    ratios = []
    for m in range(M_in.shape[0] - 1):
        a, b = M_in[m] > 0, M_in[m + 1] > 0
        cover = np.sum(a)
        ratios.append(np.sum(a & b) / cover if cover > 0 else 0.0)
    return np.array(ratios)

for nm in [40, 80, 128]:
    M_exp = mel_filterbank(nm, 64, 16000)
    r = overlap_ratio(M_exp)
    print(f'n_mels={nm:3d}  mean neighbor overlap ratio = {r.mean():.2f}')

## Wrap-up

You can now build the DFT matrix `W = exp(-2πi·kn/N)` explicitly, run `W @ x` and compare it numerically with `fft(x)` — the two agree exactly. Each row of the mel matrix `M` is a triangular filter, and `M @ p` equals the output of `mel_filterbank()` on the power spectrum vector `p`. In the Audio Core these two transforms are wrapped as `fft()` and `mel_filterbank()`, and underneath both are weighted dot products. Next lesson: **L22**, derivative basics — starting from Δy/Δx, laying the mathematical foundation for gradient descent and backpropagation.

In [ ]:
# ── TODO exercise 1: build the DFT matrix yourself ──────────────────────────────────────────
# Don't call any existing fft or np.fft.*; build the N×N matrix with only np.exp / np.arange
# Acceptance criterion: the asserts in the commented-out check below are the only standard — pass them and you're done; there is no separate "model answer" file.
def build_dft_matrix(N: int) -> np.ndarray:
    """Return the N×N DFT matrix W, satisfying W @ x == np.fft.fft(x)."""
    # TODO: fill in your implementation (hint: W[k,n] = exp(-2πi·k·n/N), built in one line with np.outer)
    raise NotImplementedError('Please implement build_dft_matrix')

# Uncomment to run the check
# x_test = np.random.default_rng(42).standard_normal(8)
# try:
#     W_test = build_dft_matrix(8)
#     assert np.allclose(W_test @ x_test, np.fft.fft(x_test), atol=1e-10), \
#         f'Values disagree! Max error: {np.max(np.abs(W_test @ x_test - np.fft.fft(x_test))):.2e}'
#     print('✅ build_dft_matrix check passed')
# except (NotImplementedError, TypeError):
#     print('⏳ Not implemented yet')


# ── TODO exercise 2: build a 3-filter mel matrix by hand ─────────────────────────────────
# N_fft=16, N_mels=3, bins=0..8; build the triangular filter matrix with the linspace-endpoint method
#
# About the "simple" in the function name: the filter centers here are equally spaced on a **linear frequency scale**
# (np.linspace applied directly to bin numbers), unlike Aurora's real mel_filterbank(), which first
# converts frequency to the mel scale (hz_to_mel), spaces the points evenly there, and finally converts back to Hz. In other words,
# "simple" = skipping the mel-scale conversion and practising only the operation "build a triangular filter matrix from N_mels+2 endpoints"
# itself — the shape and construction logic match the simplified mel demo earlier in this lesson, just with
# N_fft changed from 8 to 16.
# The acceptance criterion is again the commented-out asserts below: pass them and you're done; no need to compare numbers with mel_filterbank()
# (the scales differ, so they would never match exactly anyway). If you hit "all-zero rows", see the
# "+1e-9 and all-zero rows" note after that simplified mel demo — it's usually degeneration from n_mels being too large relative to n_fft, not a bug in your code.
def build_simple_mel_matrix(n_mels: int, n_fft: int) -> np.ndarray:
    """Return a (n_mels, n_fft//2+1) linear-scale triangular filter matrix."""
    # TODO: fill in your implementation (follow the fixed logic of the simplified mel demo earlier in this lesson)
    raise NotImplementedError('Please implement build_simple_mel_matrix')

# Uncomment to run the check
# try:
#     M_test = build_simple_mel_matrix(3, 16)
#     print('M_test shape:', M_test.shape)
#     assert M_test.shape == (3, 9), f'expected (3, 9), got {M_test.shape}'
#     assert M_test[0].sum() > 0, 'filter 0 should not be all zeros'
#     print('✅ build_simple_mel_matrix check passed')
# except (NotImplementedError, TypeError):
#     print('⏳ Not implemented yet')


---
⬇️ **Pass check**: once you've read the wrap-up, finish the whiteboard challenge below before ticking the self-review.


## ✏️ Whiteboard challenge: the DFT matrix by hand (target: 8 minutes)

Cover the screen and answer on paper:

**Question 1**: for the N=8 DFT matrix, W[2,3] = exp(?) = ?
(Hint: W[k,n] = exp(-2πi·kn/N); compute the exponent)

**Question 2**: the DFT matrix W (N=8) satisfies W·Wᴴ = ? (what multiple of the identity matrix is it?)

**Question 3**: the mel matrix M has shape (80, 1025). The input power spectrum `p` has shape (1025,).
What is the output shape of `M @ p`? What does each output value represent?

**Question 4**: what is the one-line implementation of `build_dft_matrix(N)`?
(Hint: do it in one line with `np.outer`)

Once you've worked them out, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

N = 8

# Q1: W[2,3] = exp(-2πi·2·3/8)
k, n = 2, 3
w_23 = np.exp(-2j * np.pi * k * n / N)
# exp(-2πi·6/8) = exp(-3πi/2) = 0 + i·1 = j
assert np.isclose(w_23.real, 0.0, atol=1e-12) and np.isclose(w_23.imag, 1.0, atol=1e-12), \
    f"W[2,3] should be j, got {w_23}"
print(f"Q1 ✅  W[2,3] = exp(-2πi·6/8) = {w_23:.4f}  (= +j)")

# Q2: W·Wᴴ = N·I
k_idx = np.arange(N)[:, None]
n_idx = np.arange(N)[None, :]
W = np.exp(-2j * np.pi * k_idx * n_idx / N)
prod = W @ W.conj().T
assert np.allclose(prod, N * np.eye(N), atol=1e-10), f"W·Wᴴ should be {N}·I"
print(f"Q2 ✅  W·Wᴴ = {N}·I  (the DFT matrix is √N times a unitary matrix)")

# Q3: shape of the mel matrix
n_mels, n_fft = 80, 2048
n_freq = n_fft // 2 + 1   # 1025
M_shape = (n_mels, n_freq)
p_shape = (n_freq,)
out_shape = (n_mels,)
assert M_shape[1] == p_shape[0]
print(f"Q3 ✅  M @ p: ({n_mels},{n_freq}) × ({n_freq},) = ({n_mels},)  each value = the energy in that mel band")

# Q4: build_dft_matrix
q4_ok = False
try:
    W_built = build_dft_matrix(N)
    x_test = np.random.default_rng(42).standard_normal(N)
    assert np.allclose(W_built @ x_test, np.fft.fft(x_test), atol=1e-10), \
        "build_dft_matrix disagrees with np.fft.fft"
    q4_ok = True
    print(f"Q4 ✅  build_dft_matrix(8) implemented correctly, agrees with np.fft.fft")
except (NotImplementedError, TypeError):
    print("⬜ Q4: implement build_dft_matrix() first, then run the answer-check cell")

if q4_ok:
    print("\n🎉 DFT matrix whiteboard challenge passed! You've internalized the idea FFT = matrix multiplication.")
else:
    print("\n💡 Q1–Q3 checked; finish build_dft_matrix in the exercise cell, then rerun this cell to clear Q4.")

In [ ]:
# ✏️ Self-review for this lesson
l21_review = {
    "dft_matrix_formula":         None,  # remember W[k,n] = exp(-2πi·kn/N)? True/False
    "build_dft_matrix_done":      None,  # build_dft_matrix implemented and passes its asserts? True/False
    "mel_matrix_shape_clear":     None,  # understand the mel matrix shape (n_mels, n_freq)? True/False
    "unitary_property":           None,  # understand W·Wᴴ = N·I (the unitary property)? True/False
    "whiteboard_passed":          None,  # finished the whiteboard challenge on paper? True/False
}

unfilled = [k for k, v in l21_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l21_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L21 complete! On to L22: derivatives')

---

→ **Next lesson** · [L22 · Derivatives](../3_calculus/L22_derivatives.ipynb)

> Next lesson covers **derivatives**: tangent slope, the limit definition, numerical vs analytical differentiation.